In [ ]:
# setup
!pip install SoccerNet
from google.colab import drive
drive.mount('/content/drive')

Clip creator code.

clips - (15,60,224,398,3) numpy.ndarray

labels - (5,3) numpy.ndarray

In [ ]:
# don't forget:
# !pip install SoccerNet
from SoccerNet.Downloader import SoccerNetDownloader
from SoccerNet.utils import getListGames
from moviepy.editor import VideoFileClip
import json
import random
import cv2
import os
import numpy as np

def extract_frames(video_path):
    video = cv2.VideoCapture(video_path)
    frames = []
    count = 0
    while count < frame_number:
        ret, frame = video.read()
        if not ret:
            break
        frames.append(frame)
        count += 1
    video.release()
    return frames


def delete_all_matches():
  if delete_matches:
    for i in range(number_of_videos):
      try:
          os.remove(filePath + list_games[i] + "/1.mkv")
          os.remove(filePath + list_games[i] + "/Labels.json")
      except FileNotFoundError:
          continue

def get_labels_from_time(time):
  labels = []
  #goal_frames = [time * fps for time in times if time != -1] + [time * fps + 1 for time in times if time != -1]
  goal_frames = [time * fps, time * fps + 1]
  for i in range(frame_number):
    if i in goal_frames:
      labels.append(1)
    else:
      labels.append(0)
  return labels

delete_matches = False
filePath = "drive/MyDrive/videos/"
list_games = getListGames(split="v1")
frame_number = 60
number_of_videos = 70
video_length_sec = 2700
fps = 2
clip_duration = 30
total_videos = len(list_games)
total_clips_per_game = 3
goal_time_lst = []
random.seed(0)
np.random.seed(0)
mySoccerNetDownloader = SoccerNetDownloader(LocalDirectory=filePath)
mySoccerNetDownloader.password = "s0cc3rn3t"
redoClips = True
if redoClips:
  for i in range(number_of_videos):
    goal_time_lst.append([])
    mySoccerNetDownloader.downloadGameIndex(i, files=["1.mkv", "Labels.json"])
    video = VideoFileClip(filePath + list_games[i] + "/1.mkv")
    with open(filePath + list_games[i] + "/Labels.json", 'r') as file:
      data = json.loads(file.read())
    soccer_ball_times = []
    for annotation in data["annotations"]:
      if annotation["label"] == "soccer-ball" and annotation["gameTime"][0] == "1":
        game_time = annotation["gameTime"].split(" - ")[1]
        soccer_ball_times.append(60*int(game_time.split(":")[0]) + int(game_time.split(":")[1]))

    j = 0
    for goal in soccer_ball_times:
      j += 1
      output_filename = f"game_{i+1},clip_{j}.mp4"
      sec_before_goal = random.randint(5, 25)
      goal_time_lst[-1].append(sec_before_goal)
      start_time = goal - sec_before_goal
      clip = video.subclip(start_time, start_time + clip_duration)
      clip = clip.set_duration(clip_duration).set_fps(fps)
      clip.write_videofile(filePath + output_filename, fps=fps, audio=False)

    for not_goal in range(max(0, total_clips_per_game - len(soccer_ball_times))):
      j += 1
      output_filename = f"game_{i+1},clip_{j}.mp4"
      overlap = True
      while overlap:
        start_time = random.randint(0, video_length_sec)
        for goal in soccer_ball_times:
          if goal + 30 >= start_time >= goal - 30:
            continue
        overlap = False
      goal_time_lst[-1].append(-1)
      clip = video.subclip(start_time, start_time + clip_duration)
      clip = clip.set_duration(clip_duration).set_fps(fps)
      clip.write_videofile(filePath + output_filename, fps=fps, audio=False)
    video.close()

  with open(filePath + 'goal_time_lst.json', 'w') as file:
      json.dump(goal_time_lst, file)
  delete_all_matches()

with open(filePath + 'goal_time_lst.json', 'r') as file:
    goal_time_lst = json.load(file)
goal_time_lst = goal_time_lst[:70]
clips = []
for i in range(number_of_videos):
  for j in range(total_clips_per_game):
    clips.append(extract_frames(filePath + f"game_{i+1},clip_{j+1}.mp4"))
    print(f"game_{i+1},clip_{j+1}.mp4")
clips = np.array(clips)
#goal_times = np.array(goal_time_lst)
print(goal_time_lst)
#labels = [get_labels_from_time(goal_time) for goal_time in goal_time_lst]
labels = []
for i in range(len(goal_time_lst)):
  for j in range(total_clips_per_game):
    labels.append(get_labels_from_time(goal_time_lst[i][j]))
labels = np.array(labels)
print(labels.shape)
print(clips.shape)

# random_indices = np.random.choice(number_of_videos * total_clips_per_game, size=390, replace=False)
# random_indices.sort()
# eval_clips = clips[random_indices]
# eval_labels = labels[random_indices]
# clips = clips[np.setdiff1d(np.arange(number_of_videos * total_clips_per_game), random_indices)]
# labels = labels[np.setdiff1d(np.arange(number_of_videos * total_clips_per_game), random_indices)]

In [ ]:
from SoccerNet.Downloader import SoccerNetDownloader
from SoccerNet.utils import getListGames
from moviepy.editor import VideoFileClip
import json
import random
import cv2
import os
import numpy as np
filePath = "drive/MyDrive/videos/"
list_games = getListGames(split="v1")
for i in range(0, 499):
    try:
        os.remove(filePath + list_games[i] + "/1.mkv")
        os.remove(filePath + list_games[i] + "/Labels.json")
    except FileNotFoundError:
        continue

In [ ]:
import pickle
#np.save(filePath + 'clips.npy', clips)
#np.save(filePath + 'clabels.npy', labels)
os.path.getsize(filePath + 'clips.npy')
# with open(filePath + 'clips.pkl', 'wb') as f:
#     pickle.dump(clips, f)
# with open(filePath + 'eval_clips.pkl', 'wb') as f:
#     pickle.dump(eval_clips, f)
# with open(filePath + 'labels.pkl', 'wb') as f:
#     pickle.dump(labels, f)
# with open(filePath + 'eval_labels.pkl', 'wb') as f:
#     pickle.dump(eval_labels, f)
# with open('clips.pkl', 'rb') as f:
#     clips = pickle.load(f)
# with open('eval_clips.pkl', 'rb') as f:
#     eval_clips = pickle.load(f)
# with open('labels.pkl', 'rb') as f:
#     labels = pickle.load(f)
# with open('eval_labels.pkl', 'rb') as f:
#     eval_labels = pickle.load(f)

In [ ]:
from SoccerNet.Downloader import SoccerNetDownloader
from SoccerNet.utils import getListGames
from moviepy.editor import VideoFileClip
import json
import random
import cv2
import os
import numpy as np
filePath = "drive/MyDrive/videos/"
clips = np.load(filePath + 'clips.npy')
labels = np.load(filePath + 'clabels.npy')

In [ ]:
indices = np.random.permutation(clips.shape[0])

# Shuffle both arrays according to the generated indices
clips = clips[indices]
labels = labels[indices]
eval_clips = clips[834:]
eval_labels = labels[834:]
clips = clips[:834]
labels = labels[:834]

In [ ]:
eval_clips = clips
eval_labels = labels

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms.functional as TF

def pad_and_crop(frame, target_size=(224, 224)):
    # Padding code (unchanged)
    _, _, h, w = frame.shape
    diff = abs(h - w)

    if h > w:
        padding = (diff // 2, 0, diff - (diff // 2), 0)
    else:
        padding = (0, diff // 2, 0, diff - (diff // 2))

    padded_frame = TF.pad(frame, padding)
    cropped_frame = TF.center_crop(padded_frame, target_size)

    # Convert to float and normalize (Expected for EfficientNet)
    cropped_frame = cropped_frame.float()  # Convert to float32
    cropped_frame = cropped_frame / 255.0  # Normalize pixel values to [0, 1]

    mean = [0.485, 0.456, 0.406]  # EfficientNet ImageNet mean
    std = [0.229, 0.224, 0.225]   # EfficientNet ImageNet std
    transform = TF.normalize(cropped_frame, mean, std)

    return transform


class EfficientNetLSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(EfficientNetLSTMModel, self).__init__()

        # Load the pretrained EfficientNet model
        self.efficientnet = models.efficientnet_b0(pretrained=True)

        # Remove the final classification layer from EfficientNet
        self.efficientnet.classifier = nn.Identity()

        # Define the LSTM
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, bidirectional=True)

        # Fully connected layer for final classification (goal/no goal)
        self.fc = nn.Linear(hidden_size * 2, output_size)  # Multiply by 2 for bidirectional LSTM

    def forward(self, x):
        # Input x: (batch_size, sequence_length, height, width, channels)
        batch_size, sequence_length, _, _, _ = x.shape

        # Permute to match EfficientNet input format: (batch_size, sequence_length, channels, height, width)
        x = x.permute(0, 1, 4, 2, 3)

        features = []
        for i in range(sequence_length):
            frame = x[:, i, :, :, :]  # Extract each frame (batch_size, channels, height, width)

            # Pad and resize to 224x224
            frame = pad_and_crop(frame, target_size=(224, 224))

            # Pass frame through EfficientNet
            frame_features = self.efficientnet(frame)
            features.append(frame_features)

        # Stack features to create a sequence tensor: (batch_size, sequence_length, feature_size)
        features = torch.stack(features, dim=1)

        # Pass the sequence of features through the LSTM
        lstm_out, _ = self.lstm(features)

        # Use the output of the last time step for classification
        final_output = self.fc(lstm_out)

        return final_output  # Predicted probabilities or logits

def clip_label_list(clips, labels):
  labeled_clips = []
  for i in range(len(clips)):
    labeled_clips.append((clips[i], labels[i]))
  return labeled_clips


class WeightedBCELoss(nn.Module):
    def __init__(self, weight_fn, weight_fp):
        super(WeightedBCELoss, self).__init__()
        self.weight_fn = weight_fn
        self.weight_fp = weight_fp

    def forward(self, inputs, targets):
        bce_loss = nn.BCELoss(reduction='none')(inputs, targets)
        # Apply weights: FN gets more weight, FP gets less
        weights = targets * self.weight_fn + (1 - targets) * self.weight_fp
        return (weights * bce_loss).mean()




In [ ]:
from torch.utils.data import DataLoader

hidden_sizes = [512]
num_layers_options = [3]
lr_options = [0.00001]
weight_fn_options = [70.0]
num_epochs_options = [16]
best_fp = 400
best_fn = 30
input_size = 1280
output_size = 1
batch_size = 8
for _ in range(1):  # Number of random combinations to try
    hidden_size = random.choice(hidden_sizes)
    num_layers = random.choice(num_layers_options)
    lr = random.choice(lr_options)
    weight_fn = random.choice(weight_fn_options)
    num_epochs = random.choice(num_epochs_options)
    print(f"Trying with hidden_size={hidden_size}, num_layers={num_layers}, lr={lr}, weight_fn={weight_fn}, num_epochs={num_epochs}")
    # hidden_size = 512
    # num_layers = 1
    # lr = 0.0005
    # weight_fn = 35.0
    # num_epochs = 1

    #model = torch.load(filePath + 'model_28_360.pth')
    labeled_clips = clip_label_list(clips, labels)
    labeled_eval_clips = clip_label_list(eval_clips, eval_labels)
    #model = EfficientNetLSTMModel(input_size, hidden_size, num_layers, output_size)
    model = torch.load(filePath + 'model_43_169.pth')
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay = 0.0001)

    loss_fn = WeightedBCELoss(weight_fn, weight_fp=1.0)
    loss_eval = WeightedBCELoss(weight_fn, weight_fp=1.0)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    #Freeze layers 0 to 3 of efficientNet, we don't train them:

    for name, param in model.efficientnet.named_parameters():
        if "features" in name and int(name.split('.')[1]) <= 3:
            param.requires_grad = False
    eval_loader = DataLoader(labeled_eval_clips, batch_size=210, shuffle=False)



    # Training loop
    for epoch in range(num_epochs):
        print("starting epoch " + str(epoch + 1))
        model.train()  # Set the model to training mode

        # TODO: random batching (and remember to keep the corresponing labels)
        train_loader = DataLoader(labeled_clips, batch_size=batch_size, shuffle=True)
        cnt = 0
        #print("train loader is: " + str(train_loader))
        """
        for batch in train_loader:  # train_loader provides (clips, labels) batches
            b_clips, b_labels = batch
            b_clips = b_clips.to(device)
            b_labels = b_labels.to(device)
            b_labels = b_labels.unsqueeze(-1).float()
            #print("labels shape: " + str(labels.shape))
            #print("clips shape: " + str(clips.shape))
            optimizer.zero_grad()  # Clear previous gradients

            # Step 1: Make predictions using the model
            output = model(b_clips)  # Calls the forward function
            output = torch.sigmoid(output)
            cnt += batch_size
            if cnt % (20 * batch_size) == 0:
                print("Finished " + str(cnt) + " clips out of " + str(len(clips)))
            #print("labels is: " + str(labels) + "   with shape: " + str(labels.shape))
            # Step 2: Compute the loss
            loss = loss_fn(output, b_labels)

            # Step 3: Backward pass to compute gradients
            loss.backward()

            # Step 4: Optimization step (update the weights of both EfficientNet and LSTM)
            optimizer.step()

        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')
        """
        model.eval()
        predictions = []
        for batch in eval_loader:
            b_clips, b_labels = batch
            b_clips = b_clips.to(device)
            b_labels = b_labels.to(device)
            b_labels = b_labels.unsqueeze(-1).float()
            with torch.no_grad():
                output = model(b_clips)
                output_loss = torch.sigmoid(output)
                preds = (torch.sigmoid(output) > 0.5).long()
                predictions.append(preds)
            loss_e = loss_eval(output_loss, b_labels)
            tp = 0
            fn = 0
            fp = 0
            tn = 0
            print(f'Eval Loss: {loss_e.item():.4f}')
            has_goal_pred_goal = 0
            has_goal_pred_no_goal = 0
            has_no_goal_pred_goal = 0
            has_no_goal_pred_no_goal = 0
            for i in range(len(b_labels)):
                if 1 in b_labels[i]:
                    if 1 in preds[i]:
                        has_goal_pred_goal += 1
                    else:
                        has_goal_pred_no_goal += 1
                else:
                    if 1 in preds[i]:
                         has_no_goal_pred_goal += 1
                    else:
                         has_no_goal_pred_no_goal += 1
            for i in range(len(b_labels)):
                for j in range(len(b_labels[0])):
                    if b_labels[i][j].item() == 1:
                        if preds[i][j].item() == 1:
                            tp += 1
                        else:
                            fn += 1
                    else:
                        if preds[i][j].item() == 1 and 1 not in b_labels[i,max(0,j-5):min(j+5, len(b_labels[0]))]:
                            fp += 1
                        else:
                            tn += 1
            print(tp, fn, fp, tn)
            if fn < best_fn and fp < best_fp:
                print("saving model!")
                torch.save(model, filePath + f"model_{fn}_{fp}.pth")
            print(has_goal_pred_goal, has_goal_pred_no_goal, has_no_goal_pred_goal, has_no_goal_pred_no_goal)





In [ ]:
torch.save(model, filePath + f"model_{fn}_{fp}.pth")

1. Python Flask app
2. Fine tuning
3. Get video -> run model -> clip of goal(s)
4. Write PDF & PP
5. Check different video lengths

In [ ]:
for epoch in range(num_epochs):
        print("starting epoch " + str(epoch + 1))
        model.train()  # Set the model to training mode

        # TODO: random batching (and remember to keep the corresponing labels)
        train_loader = DataLoader(labeled_clips, batch_size=batch_size, shuffle=True)
        cnt = 0
        #print("train loader is: " + str(train_loader))
        for batch in train_loader:  # train_loader provides (clips, labels) batches
            b_clips, b_labels = batch
            b_clips = b_clips.to(device)
            b_labels = b_labels.to(device)
            b_labels = b_labels.unsqueeze(-1).float()
            #print("labels shape: " + str(labels.shape))
            #print("clips shape: " + str(clips.shape))
            optimizer.zero_grad()  # Clear previous gradients

            # Step 1: Make predictions using the model
            output = model(b_clips)  # Calls the forward function
            output = torch.sigmoid(output)
            cnt += batch_size
            if cnt % (20 * batch_size) == 0:
                print("Finished " + str(cnt) + " clips out of " + str(len(clips)))
            #print("labels is: " + str(labels) + "   with shape: " + str(labels.shape))
            # Step 2: Compute the loss
            loss = loss_fn(output, b_labels)

            # Step 3: Backward pass to compute gradients
            loss.backward()

            # Step 4: Optimization step (update the weights of both EfficientNet and LSTM)
            optimizer.step()

        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')
        model.eval()
        predictions = []
        for batch in eval_loader:
            b_clips, b_labels = batch
            b_clips = b_clips.to(device)
            b_labels = b_labels.to(device)
            b_labels = b_labels.unsqueeze(-1).float()
            with torch.no_grad():
                output = model(b_clips)
                output_loss = torch.sigmoid(output)
                preds = (torch.sigmoid(output) > 0.5).long()
                predictions.append(preds)
            loss_e = loss_eval(output_loss, b_labels)
            tp = 0
            fn = 0
            fp = 0
            tn = 0
            print(f'Eval Loss: {loss_e.item():.4f}')
            has_goal_pred_goal = 0
            has_goal_pred_no_goal = 0
            has_no_goal_pred_goal = 0
            has_no_goal_pred_no_goal = 0
            for i in range(len(b_labels)):
                if 1 in b_labels[i]:
                    if 1 in preds[i]:
                        has_goal_pred_goal += 1
                    else:
                        has_goal_pred_no_goal += 1
                else:
                    if 1 in preds[i]:
                         has_no_goal_pred_goal += 1
                    else:
                         has_no_goal_pred_no_goal += 1
            for i in range(len(b_labels)):
                for j in range(len(b_labels[0])):
                    if b_labels[i][j].item() == 1:
                        if preds[i][j].item() == 1:
                            tp += 1
                        else:
                            fn += 1
                    else:
                        if preds[i][j].item() == 1 and 1 not in b_labels[i,max(0,j-5):min(j+5, len(b_labels[0]))]:
                            fp += 1
                        else:
                            tn += 1
            print(tp, fn, fp, tn)
            if fn < best_fn and fp < best_fp:
                print("saving model!")
                torch.save(model, filePath + f"model_{fn}_{fp}.pth")
            print(has_goal_pred_goal, has_goal_pred_no_goal, has_no_goal_pred_goal, has_no_goal_pred_no_goal)


In [ ]:
def extract_frames(video_path):
    video = cv2.VideoCapture(video_path)
    frames = []
    count = 0
    while count < 120:
        ret, frame = video.read()
        if not ret:
            break
        frames.append(frame)
        count += 1
    video.release()
    return frames

def get_labels_from_time(time):
  labels = []
  #goal_frames = [time * fps for time in times if time != -1] + [time * fps + 1 for time in times if time != -1]
  goal_frames = [time * fps, time * fps + 1]
  for i in range(120):
    if i in goal_frames:
      labels.append(1)
    else:
      labels.append(0)
  return labels
goal_time_lst = []
for i in range(5):
    goal_time_lst.append([])
    mySoccerNetDownloader.downloadGameIndex(i, files=["1.mkv", "Labels.json"])
    video = VideoFileClip(filePath + list_games[i] + "/1.mkv")
    with open(filePath + list_games[i] + "/Labels.json", 'r') as file:
      data = json.loads(file.read())
    soccer_ball_times = []
    for annotation in data["annotations"]:
      if annotation["label"] == "soccer-ball" and annotation["gameTime"][0] == "1":
        game_time = annotation["gameTime"].split(" - ")[1]
        soccer_ball_times.append(60*int(game_time.split(":")[0]) + int(game_time.split(":")[1]))

    j = 0
    for goal in soccer_ball_times:
      j += 1
      output_filename = f"1min_game_{i+1},clip_{j}.mp4"
      sec_before_goal = random.randint(5, 25)
      goal_time_lst[-1].append(sec_before_goal)
      start_time = goal - sec_before_goal
      clip = video.subclip(start_time, start_time + 60)
      clip = clip.set_duration(60).set_fps(fps)
      clip.write_videofile(filePath + output_filename, fps=fps, audio=False)

    for not_goal in range(max(0, total_clips_per_game - len(soccer_ball_times))):
      j += 1
      output_filename = f"1min_game_{i+1},clip_{j}.mp4"
      overlap = True
      while overlap:
        start_time = random.randint(0, video_length_sec)
        for goal in soccer_ball_times:
          if goal + 60 >= start_time >= goal - 60:
            continue
        overlap = False
      goal_time_lst[-1].append(-1)
      clip = video.subclip(start_time, start_time + 60)
      clip = clip.set_duration(60).set_fps(fps)
      clip.write_videofile(filePath + output_filename, fps=fps, audio=False)
    video.close()

with open(filePath + 'goal_time_lst_1min.json', 'w') as file:
    json.dump(goal_time_lst, file)
with open(filePath + 'goal_time_lst_1min.json', 'r') as file:
    goal_time_lst = json.load(file)
clips_1min = []
for i in range(5):
    for j in range(total_clips_per_game):
        clips_1min.append(extract_frames(filePath + f"1min_game_{i+1},clip_{j+1}.mp4"))
        print(f"1min_game_{i+1},clip_{j+1}.mp4")
clips_1min = np.array(clips_1min)
#goal_times = np.array(goal_time_lst)
print(goal_time_lst)
#labels = [get_labels_from_time(goal_time) for goal_time in goal_time_lst]
labels_1min = []
for i in range(len(goal_time_lst)):
  for j in range(total_clips_per_game):
    labels_1min.append(get_labels_from_time(goal_time_lst[i][j]))
labels_1min = np.array(labels_1min)
print(labels_1min.shape)
print(clips_1min.shape)
labeled_1min_clips = clip_label_list(clips_1min, labels_1min)

onemin_loader = DataLoader(labeled_1min_clips, batch_size=15, shuffle=False)
model.eval()
predictions = []
for batch in onemin_loader:
    b_clips, b_labels = batch
    b_clips = b_clips.to(device)
    b_labels = b_labels.to(device)
    b_labels = b_labels.unsqueeze(-1)
    with torch.no_grad():
        output = model(b_clips)
        preds = (torch.sigmoid(output) > 0.5).long()
        predictions.append(preds)
    print(preds.shape)
    print(b_labels.shape)
    tp = 0
    fn = 0
    fp = 0
    tn = 0
    alone = 0
    has_goal_pred_goal = 0
    has_goal_pred_no_goal = 0
    has_no_goal_pred_goal = 0
    has_no_goal_pred_no_goal = 0
    for i in range(len(b_labels)):
        if 1 in b_labels[i]:
            if 1 in preds[i]:
                has_goal_pred_goal += 1
            else:
                has_goal_pred_no_goal += 1
        else:
            if 1 in preds[i]:
                has_no_goal_pred_goal += 1
            else:
                has_no_goal_pred_no_goal += 1
    for i in range(len(b_labels)):
        for j in range(len(b_labels[0])):
            if b_labels[i][j].item() == 1:
                if preds[i][j].item() == 1:
                    tp += 1
                else:
                    fn += 1
            else:
                if preds[i][j].item() == 1 and 1 not in b_labels[i,max(0,j-5):min(j+5, len(b_labels[0]))]:
                    fp += 1
                if preds[i][j].item() == 1 and 1 not in b_labels[i,max(0,j-8):min(j+8, len(b_labels[0]))]:
                    alone += 1
                else:
                    tn += 1
print(tp, fn, fp, tn, alone)
print(has_goal_pred_goal, has_goal_pred_no_goal, has_no_goal_pred_goal, has_no_goal_pred_no_goal)
print(labels[0])

In [ ]:
torch.save(model, filePath + 'model_28_360.pth')

In [ ]:
# don't forget:
# !pip install SoccerNet
from SoccerNet.Downloader import SoccerNetDownloader
from SoccerNet.utils import getListGames
from moviepy.editor import VideoFileClip
import json
import random
import cv2
import os
import numpy as np

def extract_frames(video_path):
    video = cv2.VideoCapture(video_path)
    frames = []
    count = 0
    while count < frame_number:
        ret, frame = video.read()
        if not ret:
            break
        frames.append(frame)
        count += 1
    video.release()
    return frames


def delete_all_matches():
  if delete_matches:
    for i in range(number_of_videos):
      try:
          os.remove(filePath + list_games[i] + "/1.mkv")
          os.remove(filePath + list_games[i] + "/Labels.json")
      except FileNotFoundError:
          continue

def get_labels_from_time(time):
  labels = []
  #goal_frames = [time * fps for time in times if time != -1] + [time * fps + 1 for time in times if time != -1]
  goal_frames = [time * fps, time * fps + 1]
  for i in range(frame_number):
    if i in goal_frames:
      labels.append(1)
    else:
      labels.append(0)
  return labels

delete_matches = False
filePath = "drive/MyDrive/videos/"
list_games = getListGames(split="v1")
frame_number = 60
number_of_videos = 70
video_length_sec = 2700
fps = 2
clip_duration = 30
total_videos = len(list_games)
total_clips_per_game = 3
goal_time_lst = []
random.seed(0)
np.random.seed(0)
mySoccerNetDownloader = SoccerNetDownloader(LocalDirectory=filePath)
mySoccerNetDownloader.password = "s0cc3rn3t"
redoClips = False
if redoClips:
  for i in range(number_of_videos):
    goal_time_lst.append([])
    mySoccerNetDownloader.downloadGameIndex(430 + i, files=["1.mkv", "Labels.json"])
    video = VideoFileClip(filePath + list_games[430 + i] + "/1.mkv")
    with open(filePath + list_games[430 + i] + "/Labels.json", 'r') as file:
      data = json.loads(file.read())
    soccer_ball_times = []
    for annotation in data["annotations"]:
      if annotation["label"] == "soccer-ball" and annotation["gameTime"][0] == "1":
        game_time = annotation["gameTime"].split(" - ")[1]
        soccer_ball_times.append(60*int(game_time.split(":")[0]) + int(game_time.split(":")[1]))

    j = 0
    for goal in soccer_ball_times:
      j += 1
      output_filename = f"game_{i+1},clip_{j}.mp4"
      sec_before_goal = random.randint(5, 25)
      goal_time_lst[-1].append(sec_before_goal)
      start_time = goal - sec_before_goal
      clip = video.subclip(start_time, start_time + clip_duration)
      clip = clip.set_duration(clip_duration).set_fps(fps)
      clip.write_videofile(filePath + output_filename, fps=fps, audio=False)

    for not_goal in range(max(0, total_clips_per_game - len(soccer_ball_times))):
      j += 1
      output_filename = f"game_{i+1},clip_{j}.mp4"
      overlap = True
      while overlap:
        start_time = random.randint(0, video_length_sec)
        for goal in soccer_ball_times:
          if goal + 30 >= start_time >= goal - 30:
            continue
        overlap = False
      goal_time_lst[-1].append(-1)
      clip = video.subclip(start_time, start_time + clip_duration)
      clip = clip.set_duration(clip_duration).set_fps(fps)
      clip.write_videofile(filePath + output_filename, fps=fps, audio=False)
    video.close()

  with open(filePath + 'goal_time_lst.json', 'w') as file:
      json.dump(goal_time_lst, file)
  delete_all_matches()

with open(filePath + 'goal_time_lst.json', 'r') as file:
    goal_time_lst = json.load(file)
goal_time_lst = goal_time_lst[:408]
test_clips = []
for i in range(number_of_videos):
  for j in range(total_clips_per_game):
    test_clips.append(extract_frames(filePath + f"game_{i+1},clip_{j+1}.mp4"))
    print(f"game_{i+1},clip_{j+1}.mp4")
test_clips = np.array(test_clips)
#goal_times = np.array(goal_time_lst)
print(goal_time_lst)
#labels = [get_labels_from_time(goal_time) for goal_time in goal_time_lst]
test_labels = []
for i in range(len(goal_time_lst)):
  for j in range(total_clips_per_game):
    test_labels.append(get_labels_from_time(goal_time_lst[i][j]))
test_labels = np.array(test_labels)
print(test_labels.shape)
print(test_clips.shape)


In [ ]:
# setup
!pip install SoccerNet
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch

FPS = 2
FILE_PATH = "drive/MyDrive/videos/"
CLIP_DURATION = 30
FRAME_AMOUNT = FPS * CLIP_DURATION
CLIPS_PER_GAME = 3
VIDEO_LENGTH_SECS = 2700
PRODUCTION_MODEL_PATH = "model_19_335.pth"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:

from SoccerNet.Downloader import SoccerNetDownloader
from SoccerNet.utils import getListGames
from moviepy.editor import VideoFileClip
import json
import random
import cv2
import os
import numpy as np
import consts

game_list = getListGames(split="v1")
random.seed(0)
np.random.seed(0)
mySoccerNetDownloader = SoccerNetDownloader(LocalDirectory=consts.FILE_PATH)
mySoccerNetDownloader.password = "s0cc3rn3t"
number_of_videos = 408


def extract_frames(video_path):
    video = cv2.VideoCapture(video_path)
    frames = []
    for i in range(int(video.get(cv2.CAP_PROP_FRAME_COUNT))):
        ret, frame = video.read()
        if not ret:
            break
        frames.append(frame)
    video.release()
    return frames


def delete_all_matches():
    for i in range(number_of_videos):
        try:
            os.remove(consts.FILE_PATH + game_list[i] + "/1.mkv")
            os.remove(consts.FILE_PATH + game_list[i] + "/Labels.json")
        except FileNotFoundError:
            continue


def get_labels_from_time(time):
    goal_frames = {time * consts.FPS, time * consts.FPS + 1}
    return [1 if frame in goal_frames else 0 for frame in range(consts.FRAME_AMOUNT)]


def create_clips():
    goal_time_lst = []
    for i in range(number_of_videos):
        goal_time_lst.append([])
        mySoccerNetDownloader.downloadGameIndex(i, files=["1.mkv", "Labels.json"])
        video = VideoFileClip(consts.FILE_PATH + game_list[i] + "/1.mkv")
        with open(consts.FILE_PATH + game_list[i] + "/Labels.json", 'r') as labels_file:
            data = json.loads(labels_file.read())
        soccer_ball_times = []
        for annotation in data["annotations"]:
            if annotation["label"] == "soccer-ball" and annotation["gameTime"][0] == "1":
                game_time = annotation["gameTime"].split(" - ")[1]
                soccer_ball_times.append(60 * int(game_time.split(":")[0]) + int(game_time.split(":")[1]))

        j = 0
        for goal in soccer_ball_times:
            j += 1
            output_filename = f"game_{i + 1},clip_{j}.mp4"
            sec_before_goal = random.randint(5, 25)
            goal_time_lst[-1].append(sec_before_goal)
            start_time = goal - sec_before_goal
            clip = video.subclip(start_time, start_time + consts.CLIP_DURATION)
            clip = clip.set_duration(consts.CLIP_DURATION).set_fps(consts.FPS)
            clip.write_videofile(consts.FILE_PATH + output_filename, fps=consts.FPS, audio=False)

        for not_goal in range(max(0, consts.CLIPS_PER_GAME - len(soccer_ball_times))):
            j += 1
            output_filename = f"game_{i + 1},clip_{j}.mp4"
            while True:
                start_time = random.randint(0, consts.VIDEO_LENGTH_SECS)
                for goal in soccer_ball_times:
                    if goal + 30 >= start_time >= goal - 30:
                        continue
                break
            goal_time_lst[-1].append(-1)
            clip = video.subclip(start_time, start_time + consts.CLIP_DURATION)
            clip = clip.set_duration(consts.CLIP_DURATION).set_fps(consts.FPS)
            clip.write_videofile(consts.FILE_PATH + output_filename, fps=consts.FPS, audio=False)
        video.close()

    with open(consts.FILE_PATH + 'goal_time_lst.json', 'w') as created_labels_file:
        json.dump(goal_time_lst, created_labels_file)


def get_clips_and_labels(redo_clips):
    if redo_clips:
        create_clips()

    with open(consts.FILE_PATH + 'goal_time_lst.json', 'r') as file:
        goal_time_lst = json.load(file)

    goal_time_lst = goal_time_lst[:number_of_videos]
    clips = []
    for game in range(number_of_videos):
        for clip in range(consts.CLIPS_PER_GAME):
            clips.append(extract_frames(consts.FILE_PATH + f"game_{game + 1},clip_{clip + 1}.mp4"))
            print(f"game_{game + 1},clip_{clip + 1}.mp4")
    clips = np.array(clips)
    labels = []
    for i in range(len(goal_time_lst)):
        for j in range(consts.CLIPS_PER_GAME):
            labels.append(get_labels_from_time(goal_time_lst[i][j]))
    labels = np.array(labels)
    print("clips shape: " + str(clips.shape))
    print("labels shape: " + str(labels.shape))
    return clips, labels


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms.functional as TF


def pad_and_crop(frame, target_size=(224, 224)):
    # Padding code (unchanged)
    _, _, h, w = frame.shape
    diff = abs(h - w)

    if h > w:
        padding = (diff // 2, 0, diff - (diff // 2), 0)
    else:
        padding = (0, diff // 2, 0, diff - (diff // 2))

    padded_frame = TF.pad(frame, padding)
    cropped_frame = TF.center_crop(padded_frame, target_size)

    # Convert to float and normalize (Expected for EfficientNet)
    cropped_frame = cropped_frame.float()  # Convert to float32
    cropped_frame = cropped_frame / 255.0  # Normalize pixel values to [0, 1]

    mean = [0.485, 0.456, 0.406]  # EfficientNet ImageNet mean
    std = [0.229, 0.224, 0.225]  # EfficientNet ImageNet std
    transform = TF.normalize(cropped_frame, mean, std)

    return transform


class EfficientNetLSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(EfficientNetLSTMModel, self).__init__()

        # Load the pretrained EfficientNet model
        self.efficientnet = models.efficientnet_b0(pretrained=True)

        # Remove the final classification layer from EfficientNet
        self.efficientnet.classifier = nn.Identity()

        # Define the LSTM
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, bidirectional=True)

        # Fully connected layer for final classification (goal/no goal)
        self.fc = nn.Linear(hidden_size * 2, output_size)  # Multiply by 2 for bidirectional LSTM

    def forward(self, x):
        # Input x: (batch_size, sequence_length, height, width, channels)
        batch_size, sequence_length, _, _, _ = x.shape

        # Permute to match EfficientNet input format: (batch_size, sequence_length, channels, height, width)
        x = x.permute(0, 1, 4, 2, 3)

        features = []
        for i in range(sequence_length):
            frame = x[:, i, :, :, :]  # Extract each frame (batch_size, channels, height, width)

            # Pad and resize to 224x224
            frame = pad_and_crop(frame, target_size=(224, 224))

            # Pass frame through EfficientNet
            frame_features = self.efficientnet(frame)
            features.append(frame_features)

        # Stack features to create a sequence tensor: (batch_size, sequence_length, feature_size)
        features = torch.stack(features, dim=1)

        # Pass the sequence of features through the LSTM
        lstm_out, _ = self.lstm(features)

        # Use the output of the last time step for classification
        final_output = self.fc(lstm_out)

        return final_output


def clip_label_list(clips, labels):
    labeled_clips = []
    for i in range(len(clips)):
        labeled_clips.append((clips[i], labels[i]))
    return labeled_clips


class WeightedBCELoss(nn.Module):
    def __init__(self, weight_fn, weight_fp):
        super(WeightedBCELoss, self).__init__()
        self.weight_fn = weight_fn
        self.weight_fp = weight_fp

    def forward(self, inputs, targets):
        bce_loss = nn.BCELoss(reduction='none')(inputs, targets)
        # Apply weights: FN gets more weight, FP gets less
        weights = targets * self.weight_fn + (1 - targets) * self.weight_fp
        return (weights * bce_loss).mean()


In [ ]:
import torch

import consts
from dataset_creation import get_clips_and_labels
import numpy as np
from torch.utils.data import DataLoader

from model import clip_label_list, EfficientNetLSTMModel, WeightedBCELoss

train_test_split = 834
best_fp = 400
best_fn = 30
input_size = 1280
output_size = 1
batch_size = 8
hidden_size = 1024
num_layers = 2
lr = 0.0001
weight_fn = 50.0
num_epochs = 16


def data_splits(redo_clips):
    clips, labels = get_clips_and_labels(redo_clips)
    indices = np.random.permutation(clips.shape[0])
    clips = clips[indices]
    labels = labels[indices]
    eval_clips = clips[train_test_split:]
    eval_labels = labels[train_test_split:]
    clips = clips[:train_test_split]
    labels = labels[:train_test_split]
    return clips, labels, eval_clips, eval_labels


def train(model, data):
    model.train()  # Set the model to training mode

    train_loader = DataLoader(data, batch_size=batch_size, shuffle=True)
    cnt = 0
    for batch in train_loader:
        b_clips, b_labels = batch
        b_clips = b_clips.to(consts.DEVICE)
        b_labels = b_labels.to(consts.DEVICE)
        b_labels = b_labels.unsqueeze(-1).float()
        optimizer.zero_grad()  # Clear previous gradients

        # Step 1: Make predictions using the model
        output = model(b_clips)  # Calls the forward function
        output = torch.sigmoid(output)
        cnt += batch_size
        if cnt % (20 * batch_size) == 0:
            print("Finished " + str(cnt) + " clips out of " + str(len(clips)))
        # Step 2: Compute the loss
        loss = loss_fn(output, b_labels)

        # Step 3: Backward pass to compute gradients
        loss.backward()

        # Step 4: Optimization step (update the weights of both EfficientNet and LSTM)
        optimizer.step()

        print(f'Loss: {loss.item(): .4f}')

        return model


def evaluate(model, data, labeled=True):
    eval_loader = DataLoader(data, batch_size=len(data), shuffle=False)
    model.eval()
    if labeled:
        b_clips, b_labels = next(iter(eval_loader))
        b_clips = b_clips.to(consts.DEVICE)
        b_labels = b_labels.to(consts.DEVICE)
        b_labels = b_labels.unsqueeze(-1).float()
    else:
        b_clips = next(iter(eval_loader))
        b_clips = b_clips.to(consts.DEVICE)
    with torch.no_grad():
        output = model(b_clips)
        output_loss = torch.sigmoid(output)
        predictions = (torch.sigmoid(output) > 0.5).long()
    if labeled:
        loss_e = loss_eval(output_loss, b_labels)
        stats(b_labels, predictions)
        print(f'Eval Loss: {loss_e.item():.4f}')
    return predictions


def stats(b_labels, predictions):
    global best_fp, best_fn
    tp, fn, fp, tn = 0, 0, 0, 0
    has_goal_pred_goal, has_goal_pred_no_goal, has_no_goal_pred_goal, has_no_goal_pred_no_goal = 0, 0, 0, 0
    for i in range(len(b_labels)):
        if 1 in b_labels[i]:
            if 1 in predictions[i]:
                has_goal_pred_goal += 1
            else:
                has_goal_pred_no_goal += 1
        else:
            if 1 in predictions[i]:
                has_no_goal_pred_goal += 1
            else:
                has_no_goal_pred_no_goal += 1
    for i in range(len(b_labels)):
        for j in range(len(b_labels[0])):
            if b_labels[i][j].item() == 1:
                if predictions[i][j].item() == 1:
                    tp += 1
                else:
                    fn += 1
            else:
                if predictions[i][j].item() == 1 and 1 not in b_labels[i,
                                                              max(0, j - 5):min(j + 5, len(b_labels[0]))]:
                    fp += 1
                else:
                    tn += 1
    print(f"True Positives: {tp}, False Negatives: {fn}, False Positives: {fp}, True Negatives: {tn}")
    print(has_goal_pred_goal, has_goal_pred_no_goal, has_no_goal_pred_goal, has_no_goal_pred_no_goal)
    if fn < best_fn and fp < best_fp:
        best_fn = fn
        best_fp = fp
        print("saving model!")
        torch.save(model, consts.FILE_PATH + 'model.pth')


if __name__ == "__main__":
    clips, labels, eval_clips, eval_labels = data_splits(False)

    print(
        f"Trying with hidden_size={hidden_size}, num_layers={num_layers}, lr={lr}, weight_fn={weight_fn}, num_epochs={num_epochs}")

    labeled_clips = clip_label_list(clips, labels)
    labeled_eval_clips = clip_label_list(eval_clips, eval_labels)
    model = EfficientNetLSTMModel(input_size, hidden_size, num_layers, output_size)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=0.0001)

    loss_fn = WeightedBCELoss(weight_fn, weight_fp=1.0)
    loss_eval = WeightedBCELoss(weight_fn, weight_fp=1.0)

    model = model.to(consts.DEVICE)

    # Freeze layers 0 to 3 of efficientNet, we don't train them:

    for name, param in model.efficientnet.named_parameters():
        if "features" in name and int(name.split('.')[1]) <= 3:
            param.requires_grad = False

    # Training loop
    for epoch in range(num_epochs):
        print("starting epoch " + str(epoch + 1))
        model = train(model, labeled_clips)
        evaluate(model, labeled_eval_clips)


In [ ]:
model = torch.load("videos/model_19_335", map_location=consts.DEVICE)
labeled_1min_clips = clip_label_list(clips, labels)
onemin_loader = DataLoader(labeled_1min_clips, batch_size=15, shuffle=False)
model.eval()
predictions = []
for batch in onemin_loader:
    b_clips, b_labels = batch
    b_clips = b_clips.to(device)
    b_labels = b_labels.to(device)
    b_labels = b_labels.unsqueeze(-1)
    with torch.no_grad():
        output = model(b_clips)
        preds = (torch.sigmoid(output) > 0.5).long()
        predictions.append(preds)
    print(preds.shape)
    print(b_labels.shape)
    tp = 0
    fn = 0
    fp = 0
    tn = 0
    alone = 0
    has_goal_pred_goal = 0
    has_goal_pred_no_goal = 0
    has_no_goal_pred_goal = 0
    has_no_goal_pred_no_goal = 0
    for i in range(len(b_labels)):
        if 1 in b_labels[i]:
            if 1 in preds[i]:
                has_goal_pred_goal += 1
            else:
                has_goal_pred_no_goal += 1
        else:
            if 1 in preds[i]:
                has_no_goal_pred_goal += 1
            else:
                has_no_goal_pred_no_goal += 1
    for i in range(len(b_labels)):
        for j in range(len(b_labels[0])):
            if b_labels[i][j].item() == 1:
                if preds[i][j].item() == 1:
                    tp += 1
                else:
                    fn += 1
            else:
                if preds[i][j].item() == 1 and 1 not in b_labels[i,max(0,j-5):min(j+5, len(b_labels[0]))]:
                    fp += 1
                if preds[i][j].item() == 1 and 1 not in b_labels[i,max(0,j-8):min(j+8, len(b_labels[0]))]:
                    alone += 1
                else:
                    tn += 1
print(tp, fn, fp, tn, alone)
print(has_goal_pred_goal, has_goal_pred_no_goal, has_no_goal_pred_goal, has_no_goal_pred_no_goal)
print(labels[0])